In [ ]:
# import libraries

import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr
from scipy.stats import mannwhitneyu, kruskal
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.gridspec as gridspec

# MLP Model temporal training split

In [ ]:
# 1. Load the dataset
errors_path = "./Tables/errors_summary_temporal.csv"
pop_path = "../Data/us_pop_by_state.csv"  

df = pd.read_csv(errors_path)
pop_df = pd.read_csv(pop_path)

# 2. Merge population info into the dataframe
df = df.merge(
    pop_df[['state_code', '2020_census']], 
    left_on='State', 
    right_on='state_code', 
    how='left'
).drop(columns=['state_code']).rename(columns={'2020_census': 'Population'})

df.head()

In [ ]:
# 2. Print median and standard deviation for the main error metrics
# filter out the 95% CI columns ('q025', 'q975') to keep the summary clean
numeric_cols = df.select_dtypes(include=[np.number]).columns
main_metrics = [c for c in numeric_cols if 'q025' not in c and 'q975' not in c]

summary_stats = pd.DataFrame({
    'Median': df[main_metrics].median(),
    'Std Dev': df[main_metrics].std()
}).round(3)

print("--- Median and Standard Deviation of Errors ---")
display(summary_stats)

In [ ]:
# 3. Get Top 10 Highest and Lowest Errors for a specific metric
target_metric = "wMAPE_test_mean" # Can change this 

# Sort the DataFrame based on the target metric in descending order
df_sorted = df.sort_values(by=target_metric, ascending=False)

# Highest 10 (Head of descending)
highest_10 = df_sorted.head(10)[['State', 'Cluster', 'Population', target_metric]]

# Lowest 10 (Tail of descending)
lowest_10 = df_sorted.tail(10)[['State', 'Cluster', 'Population', target_metric]].sort_values(by=target_metric, ascending=True)

print(f"\n--- Top 10 States with the HIGHEST {target_metric} ---")
display(highest_10.reset_index(drop=True))

print(f"\n--- Top 10 States with the LOWEST {target_metric} ---")
display(lowest_10.reset_index(drop=True))

In [ ]:
target_metric1 = "Coverage_95_Test"
df.sort_values(target_metric1).plot.bar(x='State', y=target_metric1, figsize=(15, 4), title=f"States ranked by {target_metric1}")

In [ ]:
# Define metrics to evaluate against population
target_metrics = [
    'wMAPE_test_mean', 
    'PeakShift_test_mean', 
    'PeakAmp_ratio_test_mean'
]

results = []
for metric in target_metrics:
    # Drop any NaN values
    clean_df = df[['Population', metric]].dropna()
    
    # Pearson 
    p_corr, p_pval = pearsonr(clean_df['Population'], clean_df[metric])
    # Spearman 
    s_corr, s_pval = spearmanr(clean_df['Population'], clean_df[metric])
    
    results.append({
        'Metric': metric,
        'Pearson r': round(p_corr, 3),
        'Pearson p-val': round(p_pval, 4),
        'Spearman ρ': round(s_corr, 3),
        'Spearman p-val': round(s_pval, 4)
    })

# Convert to DataFrame
corr_df = pd.DataFrame(results)

print("--- Correlation between Population and Forecast Metrics ---")
display(corr_df)

In [ ]:
# Metrics to compare across clusters
cluster_metrics = [
    'wMAPE_test_mean', 
    'PeakShift_test_mean', 
    'PeakAmp_ratio_test_mean',
    'WIS_Test',
    'Coverage_95_Test'
]

# 1. Compute summary statistics by Cluster
cluster_summary = df.groupby('Cluster')[cluster_metrics].agg(
    ['count', 'median', 'mean', 'std']
).round(3)

print("--- Error Metrics Summary by Cluster ---")
display(cluster_summary)

# 2. Perform Kruskal-Wallis H-test to check for statistically significant cluster differences
kw_results = []
for metric in cluster_metrics:
    # Group data by cluster for the metric
    groups = [group[metric].dropna().values for _, group in df.groupby('Cluster')]
    
    # Run Kruskal-Wallis test (non-parametric ANOVA)
    stat, p_val = kruskal(*groups)
    kw_results.append({
        'Metric': metric,
        'H-statistic': round(stat, 3),
        'p-value': round(p_val, 4),
        'Significant (p < 0.05)': p_val < 0.05
    })

kw_df = pd.DataFrame(kw_results)
print("\n--- Kruskal-Wallis Test across Clusters ---")
display(kw_df)

# 3. Visual Comparison via Boxplots

# Create layout with 6 underlying columns for flexible centering
fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(2, 6, figure=fig)

# Top row: 3 subplots
ax1 = fig.add_subplot(gs[0, 0:2])
ax2 = fig.add_subplot(gs[0, 2:4])
ax3 = fig.add_subplot(gs[0, 4:6])

# Bottom row: 2 subplots centered
ax4 = fig.add_subplot(gs[1, 1:3])
ax5 = fig.add_subplot(gs[1, 3:5])

axes = [ax1, ax2, ax3, ax4, ax5]

for i, metric in enumerate(cluster_metrics):
    sns.boxplot(data=df, x='Cluster', y=metric, ax=axes[i], palette='Set2')
    sns.stripplot(data=df, x='Cluster', y=metric, ax=axes[i], color='black', alpha=0.5, jitter=0.2)
    axes[i].set_title(f'{metric} by Cluster', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('')

plt.tight_layout()
plt.show()

1. Small negative correlation between wMAPE_test_mean and state population (states with bigger populations have smaller errors), same for the Peak amplitude error.
2. wMAPE is higher and more variable for connectors, same peak amplitude. Peakshift is almost the same, WIS is higher for Hubs. Problem: Hubs states are much less than the other 2 clusters (7 vs 14 and 28). 

# MLP Model Zero-Shot training

In [ ]:
# 1. Load the dataset
errors_path = "./Tables/errors_summary_zeroshot.csv"
pop_path = "../Data/us_pop_by_state.csv"  

df = pd.read_csv(errors_path)
pop_df = pd.read_csv(pop_path)

# 2. Merge population info into the dataframe
df = df.merge(
    pop_df[['state_code', '2020_census']], 
    left_on='State', 
    right_on='state_code', 
    how='left'
).drop(columns=['state_code']).rename(columns={'2020_census': 'Population'})

# Key metrics to analyze
target_metrics = [
    'wMAPE_mean', 
    'WIS', 
    'PeakShift_mean', 
    'PeakAmp_ratio_mean',
    'PerCapita100k_mean'
]

df.head()

In [ ]:
# Train vs. Test Split Comparison

split_summary = df.groupby('Training_split')[target_metrics].agg(['median', 'mean', 'std']).round(3)
print("--- Performance Metrics: Train vs. Test Split ---")
display(split_summary)

# Mann-Whitney U test (Non-parametric comparison between Train and Test states)
split_tests = []
for metric in target_metrics:
    train_vals = df[df['Training_split'] == 'Train'][metric].dropna()
    test_vals = df[df['Training_split'] == 'Test'][metric].dropna()
    
    stat, p_val = mannwhitneyu(train_vals, test_vals, alternative='two-sided')
    split_tests.append({
        'Metric': metric,
        'U-statistic': round(stat, 3),
        'p-value': round(p_val, 4),
        'Significant Diff (p < 0.05)': p_val < 0.05
    })

print("\n--- Train vs. Test Generalization Gap (Mann-Whitney U Test) ---")
display(pd.DataFrame(split_tests))

In [ ]:
# Cluster Analysis (Test States)

df_test = df[df['Training_split'] == 'Test'].copy()

cluster_summary = df_test.groupby('Cluster')[target_metrics].agg(['count', 'median', 'mean', 'std']).round(3)
print("\n--- Test Set Error Metrics by Cluster ---")
display(cluster_summary)

# Kruskal-Wallis H-test across clusters on Test nodes
kw_results = []
for metric in target_metrics:
    groups = [group[metric].dropna().values for _, group in df_test.groupby('Cluster')]
    if len(groups) > 1 and all(len(g) > 0 for g in groups):
        stat, p_val = kruskal(*groups)
        kw_results.append({
            'Metric': metric,
            'H-statistic': round(stat, 3),
            'p-value': round(p_val, 4),
            'Significant (p < 0.05)': p_val < 0.05
        })

print("\n--- Kruskal-Wallis Test across Clusters (Test Set Only) ---")
display(pd.DataFrame(kw_results))

# Too few states in test to do this analysis probably

In [ ]:
# Correlation between errors and state population

def get_pop_correlations(dataframe, subset_label):
    results = []
    for metric in target_metrics:
        clean_df = dataframe[['Population', metric]].dropna()
        
        p_corr, p_pval = pearsonr(clean_df['Population'], clean_df[metric])
        s_corr, s_pval = spearmanr(clean_df['Population'], clean_df[metric])
        
        results.append({
            'Split Scope': subset_label,
            'Metric': metric,
            'Pearson r': round(p_corr, 3),
            'Pearson p-val': round(p_pval, 4),
            'Spearman ρ': round(s_corr, 3),
            'Spearman p-val': round(s_pval, 4)
        })
    return pd.DataFrame(results)

# Calculate for all nodes and zero-shot test nodes specifically
corr_all = get_pop_correlations(df, "All States")
corr_test = get_pop_correlations(df[df['Training_split'] == 'Test'], "Test Set Only")

corr_df = pd.concat([corr_all, corr_test], ignore_index=True)

print("--- Zero-Shot Model: Population Correlation Analysis ---")
display(corr_df)

In [ ]:
# Visualizations

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

plot_metrics = ['wMAPE_mean', 'WIS', 'PeakShift_mean', 'PeakAmp_ratio_mean']

for i, metric in enumerate(plot_metrics):
    # Hue handles Train vs Test breakdown across clusters
    sns.boxplot(data=df, x='Cluster', y=metric, hue='Training_split', ax=axes[i], palette='Set1')
    sns.stripplot(data=df, x='Cluster', y=metric, hue='Training_split', dodge=True, 
                  ax=axes[i], color='black', alpha=0.6, legend=False)
    axes[i].set_title(f'{metric} by Cluster & Split', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('')

plt.tight_layout()
plt.show()